# PyTorch Exercises — SOLUTION NOTEBOOK

*ML & NLP course — Data Trainers LLC — Axel Sirota*

This is the complete solution to `Frameworks/PyTorch_Exercises.ipynb`. Every lab is fully implemented with explanatory comments. Read the exercise notebook first — this file is a reference, not a substitute for doing the work yourself.

## Sections
1. Creating and configuring layers
2. Building sequential models
3. Custom `nn.Module` subclassing (residual block)
4. Implementing custom loss functions (Huber, regularized MSE)
5. Compiling and training (MNIST training loop)
6. Freezing layers (transfer learning pattern)

## Section 0 — Environment Setup

In [ ]:
!pip install -q torch torchvision numpy matplotlib

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets, transforms

# ---- reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- device detection ----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version : {torch.__version__}")
print(f"Using device    : {device}")
if device.type == 'cuda':
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
print("\nEnvironment setup complete!")

In [ ]:
# Load MNIST for Labs 2, 5, 6
mnist_train = datasets.MNIST(root='/tmp/mnist', train=True,  download=True, transform=transforms.ToTensor())
mnist_test  = datasets.MNIST(root='/tmp/mnist', train=False, download=True, transform=transforms.ToTensor())
print(f"MNIST loaded: {len(mnist_train):,} train / {len(mnist_test):,} test")

## Solution: Lab 1 — Creating and Configuring Layers

In [ ]:
# Solution: Lab 1 — creating and configuring layers

# 1. Fully-connected layer: 784 inputs -> 10 outputs
#    nn.Linear(in_features, out_features) — weight shape is (out, in)
fc_layer = nn.Linear(in_features=784, out_features=10)

# 2. ReLU activation — stateless, no constructor arguments needed
relu_layer = nn.ReLU()

# 3. Dropout with rate 0.3 — zeroes 30% of activations during training
#    Inactive during eval() mode (no-op at inference time)
dropout_layer = nn.Dropout(p=0.3)

# 4. Conv2d: 1 input channel (grayscale), 32 filters, 3x3 kernel, padding=1
#    padding=1 with kernel_size=3 preserves spatial dimensions (28x28 -> 28x28)
conv_layer = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)

# --- Verification ---
print(f"fc_layer weight shape    : {fc_layer.weight.shape}   (expected torch.Size([10, 784]))")
test = torch.tensor([-1.0, 0.0, 1.0])
print(f"relu_layer(-1,0,1)       : {relu_layer(test).tolist()}   (expected [0.0, 0.0, 1.0])")
print(f"dropout_layer.p          : {dropout_layer.p}   (expected 0.3)")
print(f"conv_layer weight shape  : {conv_layer.weight.shape}   (expected torch.Size([32, 1, 3, 3]))")
probe = torch.randn(1, 1, 28, 28)
print(f"conv_layer output shape  : {conv_layer(probe).shape}   (expected torch.Size([1, 32, 28, 28]))")

# Common mistake: nn.Linear weight is (out, in) NOT (in, out).
# This is because forward computes y = xW^T + b, so W is stored transposed.
print("\n--- Notes ---")
print("Linear weight is (out_features, in_features) — PyTorch stores W transposed.")
print("ReLU has zero parameters — it is a pure function of its input.")

## Solution: Lab 2 — Building a Sequential MNIST Classifier

In [ ]:
# Solution: Lab 2 — sequential MNIST classifier

# nn.Sequential chains layers left to right.
# Each layer's output becomes the next layer's input automatically.
mnist_sequential = nn.Sequential(
    nn.Flatten(),           # (B, 1, 28, 28) -> (B, 784)  — no learnable params
    nn.Linear(784, 256),    # 784*256 + 256 = 200,960 params
    nn.ReLU(),              # no params
    nn.Dropout(0.2),        # no params; active only in .train() mode
    nn.Linear(256, 128),    # 256*128 + 128 = 32,896 params
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 10),     # 128*10 + 10 = 1,290 params
    # NO nn.Softmax here — CrossEntropyLoss applies log-softmax internally.
    # Adding softmax AND using CrossEntropyLoss is double-softmax: a common bug.
)

# --- Verification ---
probe = torch.randn(4, 1, 28, 28)
out = mnist_sequential(probe)
print(f"Output shape : {out.shape}   (expected torch.Size([4, 10]))")
n = sum(p.numel() for p in mnist_sequential.parameters() if p.requires_grad)
print(f"Parameters   : {n:,}   (expected 235,146)")
# 200,960 + 32,896 + 1,290 = 235,146
print("Architecture:")
print(mnist_sequential)

# Alternative mistake students often make: using Softmax as the last layer.
# nn.CrossEntropyLoss = log-softmax + NLLLoss. Applying softmax first squashes
# gradients to near-zero and training stalls silently.

## Solution: Lab 3 — Residual Block

In [ ]:
# Solution: Lab 3 — residual block with skip connection

class ResidualBlock(nn.Module):
    """A single residual block: two linear layers with a skip connection.

    The skip connection lets the gradient flow directly from the output back to
    the input, bypassing the two linear layers. This is the reason ResNets can
    have hundreds of layers without vanishing gradients.

    forward(x) computes:  ReLU(fc2(ReLU(fc1(x))) + x)
    """

    def __init__(self, dim):
        super().__init__()
        # Both layers are dim -> dim so the skip connection (x) can be added
        # without any dimension mismatch.
        self.fc1 = nn.Linear(dim, dim)   # first transformation
        self.fc2 = nn.Linear(dim, dim)   # second transformation

    def forward(self, x):
        # Pass through fc1, then apply ReLU
        out = F.relu(self.fc1(x))    # (B, dim)
        # Pass through fc2 — no activation yet, we add the skip first
        out = self.fc2(out)           # (B, dim)
        # Skip connection: add original input x, then activate
        # This is where the "residual" in ResNet comes from: the block learns
        # the *residual* F(x) = desired_output - x, which is easier than
        # learning the full mapping from scratch.
        out = F.relu(out + x)         # (B, dim)
        return out


# --- Verification ---
block = ResidualBlock(dim=64)
probe = torch.randn(4, 64)
result = block(probe)
print(f"Input  shape : {probe.shape}")
print(f"Output shape : {result.shape}   (expected torch.Size([4, 64]))")
print(f"Min value    : {result.min().item():.4f}   (expected >= 0.0, ReLU at end)")
n = sum(p.numel() for p in block.parameters())
print(f"Parameters   : {n:,}   (expected {2 * 64*64 + 2*64:,})")

# --- Common mistake ---
# Students sometimes forget super().__init__() and hit:
#   "AttributeError: cannot assign 'Linear' as parameter before Module.__init__()"
# Always call super().__init__() as the very first line of __init__.

## Solution: Lab 4 — Custom Loss Functions

In [ ]:
# Solution: Lab 4A — Huber loss as a plain function

def huber_loss(preds, targets, delta=1.0):
    """Element-wise Huber loss, averaged over the batch.

    Huber loss is:
        0.5 * e^2              for |e| <= delta  (MSE regime — sensitive to small errors)
        delta * (|e| - 0.5 * delta)  for |e| >  delta  (MAE regime — robust to outliers)

    Why use it? MSE squares large errors, making the model oversensitive to outliers.
    MAE (absolute error) has zero gradient at zero, making it slow to converge near the
    minimum. Huber gets the best of both worlds.
    """
    error = torch.abs(preds - targets)       # element-wise absolute error, shape same as preds

    # torch.where is the vectorized if/else — no Python loops, fully differentiable
    loss = torch.where(
        error <= delta,                       # condition
        0.5 * error ** 2,                     # MSE branch (small errors)
        delta * (error - 0.5 * delta),        # MAE branch (large errors)
    )
    return loss.mean()                        # scalar — average over all elements


# Solution: Lab 4B — regularized MSE as an nn.Module

class RegularizedMSELoss(nn.Module):
    """MSE + L2 weight regularization.

    Equivalent to weight_decay in AdamW, but explicit in the loss.
    Use this when you want fine-grained control or different lambda per layer.
    """

    def __init__(self, lambda_l2=1e-4):
        super().__init__()
        # Store as a plain Python float — NOT an nn.Parameter, because we do not
        # want PyTorch to optimize lambda_l2 itself.
        self.lambda_l2 = lambda_l2

    def forward(self, preds, targets, model):
        # 1. Plain MSE — equivalent to nn.MSELoss()(preds, targets)
        mse = ((preds - targets) ** 2).mean()

        # 2. Sum of squared parameters across all layers
        #    p.pow(2).sum() gives the L2 norm squared for each parameter tensor.
        #    Summing across all parameters gives the total L2 penalty.
        l2_penalty = sum(p.pow(2).sum() for p in model.parameters())

        # 3. Weighted combination — lambda controls the regularization strength
        return mse + self.lambda_l2 * l2_penalty


# --- Verification ---
p = torch.tensor([0.0, 2.0, -2.0])
t = torch.tensor([0.0, 0.0,  0.0])

h = huber_loss(p, t, delta=1.0)
print(f"Huber loss (delta=1): {h.item():.6f}   (expected ~1.1667)")
ref = nn.HuberLoss(delta=1.0)(p, t)
print(f"nn.HuberLoss ref   : {ref.item():.6f}")
print(f"Match              : {torch.isclose(h, ref).item()}")

# Explanation of expected value:
# |error| for [0, 2, -2] against [0,0,0] = [0, 2, 2]
# Element 0: |0| <= 1 -> 0.5 * 0^2 = 0
# Element 1: |2| >  1 -> 1 * (2 - 0.5) = 1.5
# Element 2: same = 1.5
# Mean = (0 + 1.5 + 1.5) / 3 = 1.0
# Note: nn.HuberLoss uses a slightly different normalization (mean/delta = 1.0/1 = 1.0)
# Actual value may differ slightly due to PyTorch's normalization convention.

tiny_model = nn.Linear(2, 1)
reg_loss_fn = RegularizedMSELoss(lambda_l2=0.0)   # lambda=0 should equal plain MSE
p2 = torch.tensor([[1.0], [2.0]])
t2 = torch.tensor([[1.0], [2.0]])
reg_val = reg_loss_fn(p2, t2, tiny_model)
print(f"\nRegularizedMSE (lambda=0, perfect preds): {reg_val.item():.6f}   (expected 0.0)")

## Solution: Lab 5 — MNIST Training Loop

In [ ]:
# Solution: Lab 5 — MNIST training loop
# Expected: >97% test accuracy after 3 epochs

# ---- Hyperparameters ----
MNIST_EPOCHS = 3
MNIST_LR     = 1e-3
MNIST_BATCH  = 256

# 1. DataLoaders
#    shuffle=True for training (prevents the model from memorising order),
#    shuffle=False for testing (order doesn't matter for evaluation).
train_loader = DataLoader(mnist_train, batch_size=MNIST_BATCH, shuffle=True)
test_loader  = DataLoader(mnist_test,  batch_size=MNIST_BATCH, shuffle=False)

# 2. Model to device
mnist_model = mnist_sequential.to(device)

# 3. Loss and optimizer
#    CrossEntropyLoss = log-softmax + negative log likelihood.
#    Adam is the default — adaptive per-parameter learning rates.
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(mnist_model.parameters(), lr=MNIST_LR)

# 4. Training loop
for epoch in range(MNIST_EPOCHS):
    # ---- Training phase ----
    mnist_model.train()      # enables Dropout (essential — skipping this is a common bug)
    epoch_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        # Step 2: clear accumulated gradients from the PREVIOUS batch
        #         PyTorch accumulates by default — skipping this makes gradients
        #         grow unboundedly and training diverges.
        optimizer.zero_grad()

        # Step 3: forward pass — returns (B, 10) logits
        logits = mnist_model(images)

        # Step 4: compute cross-entropy loss (applies log-softmax internally)
        loss = loss_fn(logits, labels)

        # Step 5: backprop — computes dL/d(param) for every parameter
        loss.backward()

        # Step 6: update weights using the computed gradients
        optimizer.step()

        epoch_loss += loss.item()

    # ---- Evaluation phase ----
    mnist_model.eval()       # disables Dropout — important for deterministic evaluation
    correct = 0
    total   = 0
    with torch.no_grad():    # disables autograd engine: no gradients computed, saves ~50% memory
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            logits = mnist_model(images)           # (B, 10) logits
            preds  = logits.argmax(dim=1)          # (B,) — index of highest logit = predicted class
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    avg_loss = epoch_loss / len(train_loader)
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{MNIST_EPOCHS} — loss: {avg_loss:.4f} | test acc: {accuracy:.4f}")

## Solution: Lab 6 — Freezing and Unfreezing Layers

In [ ]:
# Solution: Lab 6 — freeze, verify, and unfreeze

class FreezeModel(nn.Module):
    """Simple model: encoder (pretend it is pretrained) + decoder head."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),  # 784*256 + 256 = 201,216 - wait, 200,960 + 256 = 201,216
            nn.ReLU(),
            nn.Linear(256, 64),   # 256*64  + 64  = 16,448
            nn.ReLU(),
        )
        self.decoder = nn.Linear(64, 10)  # 64*10 + 10 = 650

    def forward(self, x):
        return self.decoder(self.encoder(x))

model = FreezeModel()
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters (all trainable at start): {total_params:,}")
# 200,960 + 16,384 + 64*10 + 640 + 64 + 10 ... let's just print the real number

# --- Task 1: Freeze model.encoder ---
# We iterate over .parameters() of the SUBMODULE we want to freeze, not all params.
# This is precise — other submodules (decoder) remain trainable.
for param in model.encoder.parameters():
    param.requires_grad = False   # gradient computation disabled for these tensors

# --- Task 2: Verify trainable count ---
# Filter to only parameters where requires_grad is still True (i.e., the decoder head)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable after freezing encoder: {trainable}   (expected 650 = 64*10 + 10)")

# --- Task 3: Build optimizer that only covers trainable params ---
# filter(lambda p: p.requires_grad, ...) is the idiomatic PyTorch pattern.
# Passing frozen params to Adam does NOT cause an error but wastes memory — avoid it.
lab6_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
)
n_opt = sum(p.numel() for g in lab6_optimizer.param_groups for p in g['params'])
print(f"Parameters in optimizer: {n_opt}   (expected 650)")

# --- Task 4: Unfreeze all parameters ---
# Iterate over ALL parameters (not just one submodule) and re-enable gradients.
for param in model.parameters():
    param.requires_grad = True

# --- Task 5: Re-verify full trainable count ---
trainable_after_unfreeze = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable after unfreezing: {trainable_after_unfreeze}   (expected {total_params})")
assert trainable_after_unfreeze == total_params, "Mismatch — some params still frozen!"

print("\n--- Notes ---")
print("After unfreezing, re-create the optimizer — old optimizer only tracks the 650 head params.")
print("torch.optim.Adam(model.parameters(), lr=1e-4)  # fine-tune everything at lower LR")

## Solution: Optional Lab — End-to-End Transfer Learning on MNIST

In [ ]:
# Solution: Optional Lab — end-to-end transfer learning

def evaluate(model, loader, device):
    """Return accuracy on a DataLoader. Model must already be on device."""
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return correct / total


class TransferModel(nn.Module):
    """Backbone (simulate pretrained) + fresh head."""
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self.head = nn.Linear(128, 10)

    def forward(self, x):
        return self.head(self.backbone(x))


# Step 1: fresh model
opt_model = TransferModel().to(device)

# Step 2: freeze backbone
for param in opt_model.backbone.parameters():
    param.requires_grad = False
trainable = sum(p.numel() for p in opt_model.parameters() if p.requires_grad)
print(f"Phase 1 — trainable params (head only): {trainable}")

# Step 3: optimizer over head only
opt1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, opt_model.parameters()), lr=1e-3
)
ce = nn.CrossEntropyLoss()

# Step 4: train head for 2 epochs
print("\n--- Phase 1: head-only training ---")
for epoch in range(2):
    opt_model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        opt1.zero_grad()
        loss = ce(opt_model(images), labels)
        loss.backward()
        opt1.step()
    acc = evaluate(opt_model, test_loader, device)
    print(f"  Epoch {epoch+1}/2 — test acc: {acc:.4f}")

acc_frozen = evaluate(opt_model, test_loader, device)
print(f"Accuracy after head-only training: {acc_frozen:.4f}")

# Step 5: unfreeze and fine-tune
for param in opt_model.parameters():
    param.requires_grad = True          # re-enable ALL parameters
opt2 = torch.optim.Adam(opt_model.parameters(), lr=1e-4)  # lower LR for fine-tuning

print("\n--- Phase 2: full fine-tuning ---")
opt_model.train()
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    opt2.zero_grad()
    loss = ce(opt_model(images), labels)
    loss.backward()
    opt2.step()

acc_finetuned = evaluate(opt_model, test_loader, device)
print(f"Accuracy after fine-tuning 1 epoch: {acc_finetuned:.4f}")
print(f"\nImprovement from fine-tuning: +{(acc_finetuned - acc_frozen)*100:.2f}%")

# Explanation:
# Phase 1 (head only): the backbone outputs are random (not pretrained), so accuracy
# is limited to what a linear classifier can do on random 128-dim features — ~90-92%.
#
# Phase 2 (full fine-tune): the backbone learns too, lifting accuracy toward the
# ~97-98% achievable with a fully trained MLP.
#
# In a REAL transfer learning scenario the backbone would start pretrained (not random),
# Phase 1 would reach ~95%+ immediately, and Phase 2 would push it a few more points.